> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production note (active)

The FastAPI backend is already library-integrated. Routes in `apps/api/routers/` import from `mrta.*`:

```python
from mrta.core.rag_pipeline import rag_query
from mrta.core.llm import LLMClient
from mrta.retrieval.vector_store import VectorStore
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 05.

# Phase 5 — FastAPI Backend

**Goal:** Wrap the RAG pipeline from Phase 4 into a clean HTTP API with three endpoints:

- `POST /upload`   — upload a PDF, parse and index it.
- `POST /ask`      — ask a question; return answer + citations.
- `GET  /documents` — list indexed documents.

Plus `/health` for readiness probes and `/docs` for the auto-generated OpenAPI UI.

This is where the project starts looking production-shaped. We use Pydantic v2 for typed request/response models, dependency injection for shared resources, and `httpx` to call the live server from this notebook.


## 5.1 — Why FastAPI?

- Type-driven (Pydantic v2) — request/response validation is free.
- Async-friendly — useful for streaming LLM responses later.
- Auto OpenAPI + Swagger UI — every endpoint gets a doc page at `/docs`.
- Production-grade ASGI server (`uvicorn`) — supports workers, hot-reload, graceful shutdown.

Alternatives considered: Flask (no native async or types), Starlette (lower level), Litestar (newer, smaller ecosystem).


## 5.2 — Pydantic request / response schemas


In [5]:
# Full implementation: see apps/api/schemas/

## 5.3 — The FastAPI app

Here is the file that lives at `apps/api/main.py`. We use FastAPI's lifespan event to load the embedder, vector store, and LLM exactly once at startup.


In [6]:
# Full implementation: see apps/api/main.py

## 5.4 — Running it locally

```bash
# From the repo root, with the venv activated:
uvicorn apps.api.main:app --reload --port 8000
```

Then visit:

- http://localhost:8000/docs — Swagger UI
- http://localhost:8000/health — should return `{"status": "ok"}`


## 5.5 — Driving the API from this notebook

Once the server is running in another terminal, the rest of this notebook talks to it over HTTP. This is exactly how the Streamlit UI in Phase 6 will talk to the backend, so it is good practice.


In [20]:
import httpx

BASE = "http://localhost:8000"

def alive() -> bool:
    try:
        r = httpx.get(f"{BASE}/health", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if alive():
    print("API is up.")
else:
    print("API is not running. Start it in another terminal:")
    print("  uvicorn app.api.main:app --reload --port 8000")


API is up.


## 5.6 — Upload a PDF


In [32]:
import os, sys
from pathlib import Path

# find the root of the repository by looking for 'pyproject.toml'
repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

In [29]:
from pathlib import Path

pdf_path = Path("data/raw/attention_is_all_you_need.pdf")
if not pdf_path.exists():
    raise FileNotFoundError("Run Notebook 01 first to download the sample PDF.")

if alive():
    with pdf_path.open("rb") as f:
        files = {"file": (pdf_path.name, f, "application/pdf")}
        r = httpx.post(f"{BASE}/upload", files=files, timeout=120)
    r.raise_for_status()
    print(r.json())
else:
    print("Skipping — server not running.")


{'doc_id': 'attention_is_all_you_need_a639448e61', 'source': 'attention_is_all_you_need.pdf', 'n_pages': 15, 'n_chunks': 61}


## 5.7 — Ask a question


In [30]:
if alive():
    payload = {"question": "What is multi-head attention and why is it useful?", "top_k": 5}
    r = httpx.post(f"{BASE}/ask", json=payload, timeout=120)
    print(r.status_code, r.text)   # see raw response first
    r.raise_for_status()
    resp = r.json()
    print("Q:", payload["question"])
    print("\nA:", resp["answer"])
    # print("\ncited:", resp["cited_pages"], " | latency:", round(resp["latency_s"], 2), "s")
    print("\ncited pages:", [s["page"] for s in resp["sources"]])
    print("latency:", round(resp["latency_s"], 2), "s")


200 {"answer":"Multi-head attention is a type of attention mechanism used in the Transformer model. It allows every position in the decoder to attend over all positions in the input sequence, mimicking typical encoder-decoder attention mechanisms in sequence-to-sequence models.\n\nAccording to the provided context, multi-head attention is useful because it enables the Transformer model to process multiple representations simultaneously and jointly attend to information from different representation subspaces at different positions. This allows the model to capture complex relationships between different parts of the input sequence.\n\nThe usefulness of multi-head attention can be inferred from its application in three different ways: \"encoder-decoder attention\" layers, where queries come from the previous decoder layer and memory keys and values come from the output of the encoder; and self-attention layers, where all keys, values, and queries come from the same place, specifically t

## 5.8 — List documents


In [31]:
if alive():
    r = httpx.get(f"{BASE}/documents")
    r.raise_for_status()
    for d in r.json():
        print(f"  {d['doc_id']:30s}  {d['source']:40s}  pages={d['n_pages']}  chunks={d['n_chunks']}")


  BLIP-2_c66987cf12               BLIP-2.pdf                                pages=13  chunks=74
  attention_is_all_you_need_a639448e61  attention_is_all_you_need.pdf             pages=15  chunks=305


## 5.9 — Production-shaped touches

Things to add when you graduate from "works on my machine":

| Concern         | Recommended addition                                                            |
|-----------------|----------------------------------------------------------------------------------|
| Auth            | `fastapi.security.APIKeyHeader` + a `Depends(verify_key)` on write endpoints.    |
| Rate limiting   | `slowapi` middleware, per-IP buckets.                                            |
| Tracing         | `opentelemetry-instrumentation-fastapi`; export to Jaeger or LangSmith.          |
| Errors          | Global exception handler that returns JSON, not HTML.                            |
| Validation      | Use `pydantic.Field(...)` ranges (we did); reject empty PDFs at upload.          |
| Streaming       | Switch `/ask` to `StreamingResponse` and stream the LLM tokens.                  |
| Background jobs | For large PDFs, return `202 Accepted` + a job_id; index in a Celery worker.      |

We will add Docker + logging in Notebook 09. Streaming is a great post-tutorial exercise.


## What you learned

- The full set of endpoints needed for a RAG app: `upload`, `ask`, `documents`, `health`.
- How to use Pydantic v2 for request/response validation.
- The lifespan pattern for loading expensive resources once.
- Driving the API from Python via `httpx`.
- A roadmap from "works on my laptop" to production-shaped.

## Exercises

1. Add a `DELETE /documents/{doc_id}` endpoint that removes a doc + its chunks (needs a vector store that supports filtering, e.g. Qdrant).
2. Convert `/ask` to a streaming endpoint that yields tokens as the LLM produces them.
3. Add a `/ask` rate limit of 30 requests/minute per IP.

## Next: [Phase 6 — Streamlit frontend](./2026-05-25-phase06-streamlit-frontend.ipynb)


# Answers to Exercises




## Exercise 1 — DELETE /documents/{doc_id}

Why FAISS can't do this: FAISS stores vectors in a flat array — there's no way to remove a row by ID. The only option is to rebuild the entire index without the deleted doc's chunks, which is expensive and fragile.

This is the architectural reason Qdrant exists. Qdrant is a proper vector database that stores vectors as named points with payloads (metadata), and supports delete_by_filter(doc_id=...) directly.

For now, with FAISS you can implement a soft approach in the notebook:


# FAISS workaround: rebuild the index without the deleted doc
def delete_doc(store, doc_id: str) -> int:
    before = len(store._chunks)
    store._chunks = [c for c in store._chunks if c.doc_id != doc_id]
    removed = before - len(store._chunks)

    # Rebuild FAISS index from remaining chunks
    import faiss
    store._index = faiss.IndexFlatIP(store._embedder.dim)
    if store._chunks:
        embs = store._embedder.embed([c.text for c in store._chunks])
        store._index.add(embs)
    return removed

n = delete_doc(store, "some_doc_id")
print(f"Removed {n} chunks")
The FastAPI endpoint would look like:


@router.delete("/documents/{doc_id}")
def delete_document(doc_id: str, store=Depends(get_store)):
    remaining = [c for c in store._chunks if c.doc_id != doc_id]
    removed = len(store._chunks) - len(remaining)
    if removed == 0:
        raise HTTPException(404, f"doc_id {doc_id!r} not found")
    # rebuild index — expensive for large stores
    store._chunks = remaining
    # ... rebuild FAISS index ...
    return {"doc_id": doc_id, "chunks_removed": removed}
The exercise note "needs Qdrant" is the honest answer — this is a known limitation of FAISS that ADR-002 in this project explicitly documents.

## Exercise 2 — Streaming /ask

Concept: currently /ask waits for the LLM to finish generating the full answer, then sends it all at once. Streaming sends each token as it's produced — the browser/client sees words appearing in real time, which feels much faster even if total latency is the same.

Ollama supports streaming via stream=True:


from fastapi.responses import StreamingResponse

@router.post("/ask/stream")
def ask_stream(req: AskRequest, store=Depends(get_store), llm=Depends(get_llm)):
    sources = store.search(req.question, k=req.top_k)
    prompt  = load_prompt("rag", chunks=sources, question=req.question)

    def token_generator():
        import ollama
        for chunk in ollama.chat(
            model=llm.model,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            stream=True,
        ):
            token = chunk["message"]["content"]
            yield token   # sends each token immediately

    return StreamingResponse(token_generator(), media_type="text/plain")
On the client side in the notebook:


with httpx.stream("POST", f"{BASE}/ask/stream", json=payload, timeout=120) as r:
    for token in r.iter_text():
        print(token, end="", flush=True)
The tradeoff: you lose the structured AskResponse (sources, latency) — you'd need a separate call or a protocol like SSE (Server-Sent Events) to send metadata alongside the stream.

## Exercise 3 — Rate limiting 30 req/min per IP

Concept: without a rate limit, a single user (or bot) can hammer the /ask endpoint and saturate your local Ollama instance. slowapi is a thin wrapper around the limits library that integrates with FastAPI as middleware.


pip install slowapi

# main.py additions
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded

limiter = Limiter(key_func=get_remote_address)   # bucket per client IP
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)
Then decorate the endpoint:


# routers/ask.py
from slowapi import Limiter
from slowapi.util import get_remote_address
from fastapi import Request

limiter = Limiter(key_func=get_remote_address)

@router.post("/ask", response_model=AskResponse)
@limiter.limit("30/minute")
def ask(request: Request, req: AskRequest, store=Depends(get_store), llm=Depends(get_llm)):
    ...
Note the request: Request parameter — slowapi needs it to extract the client IP. When the limit is exceeded it automatically returns 429 Too Many Requests with a Retry-After header.